# 04 Pupil Tracking Ingestion

Populate pupil-tracking tables and run quality checks.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os

# change to the upper level folder to detect dj_local_conf.json
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import datajoint as dj
from adamacs.pipeline import subject, session, equipment, surgery, event, trial, imaging, behavior, scan, model,  analysis, virtual_markers_optitrack, mocap
from adamacs.ingest import session as isess
from adamacs.ingest import behavior as ibe
from adamacs.utility import *
import pathlib
from natsort import natsorted, ns
import datajoint as dj1
from rspace_client.eln import eln
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from importlib.metadata import version
from adamacs.helpers import dj_helpers as djh
from adamacs.helpers import stack_helpers as sh 
from adamacs.helpers import adamacs_ingest as ai
from pathlib import Path

dj.__version__

print(dj.__version__)
print(dj.config['custom']['database.prefix'])



In [ ]:
from adamacs.schemas import pupil_tracking

Parameter table for pupil fitting.

In [ ]:
virtual_markers_optitrack.RigidMouseTracking()

In [ ]:
virtual_markers_optitrack()

Populate the table with new data

In [ ]:
# Populate GazeReconstruction3D for the target scan
scansi = "scan9FU1BEUM"
gaze_sources = (pupil_tracking.PupilRotationOptiTrack & f'scan_id = "{scansi}"')
print(f'Populating pupil_tracking.GazeReconstruction3D for {scansi} ({len(gaze_sources)}) recordings...')
pupil_tracking.GazeReconstruction3D.populate(
    gaze_sources,
    display_progress=True,
    suppress_errors=True,
)


In [ ]:
mocap.MotionCapture.RigidBodyPosition()

In [ ]:
# Populate pupil_tracking tables for a specific scan
target_scan_id = 'scan9FU1BEUM'  # adjust as needed

scan_key = (
    scan.Scan
    & f"scan_id = '{target_scan_id}'").fetch1("KEY")

In [ ]:
mocap.MotionCapture.RigidBodyPosition & scan_key

In [ ]:
xpos, ypos, zpos = (mocap.MotionCapture.RigidBodyPosition & scan_key).fetch("x_pos", "y_pos", "z_pos")
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.plot(xpos[0], ypos[0], zpos[0], label=f'Rigid Body Position ({eye} eye)', alpha=0.8)
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
ax.set_zlabel('Z Position')
ax.set_title(f'3D Rigid Body Trajectory - {scansi} ({eye} eye)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pupil_tracking.GazeReconstruction3D().delete

In [ ]:
pupil_tracking.PupilEllipseParameter.insert1({'parameter_id': 4, 'likelihood_thres': 0.2, 'exclude_ir_std': 15.0, 'ellipticity_thres': 0.75, 'description': 'new parameter sloppy'})


Look for all ingested video recordings that have "eye" in their name and have not been processed for pupil tracking yet. 
Add parameter set to be used

In [ ]:
scan_key = (scan.Scan & 'scan_id = "scan9FU07CC2"').fetch('KEY')[0]

In [ ]:
unique_recording_ids = (model.VideoRecordingNew & scan_key).fetch('recording_id')
unique_recording_ids

In [ ]:
# process_keys = (model.VideoRecordingNew & ('recording_id LIKE "%bench2p_face%"')).fetch('KEY')
# process_keys = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording * scan.ScanPath & "session_datetime >= '2024-10-02'" & "initials = 'NK'").fetch("KEY")
process_keys = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording * scan.ScanPath & scan_key).fetch("KEY")

for key in process_keys:
    key.update({'parameter_id': 4})

In [ ]:
session.Session  & process_keys

In [ ]:
scansi = "scan9FU1BEUM"	
eye = 'left'
scan_key_left = (pupil_tracking.PupilRotationOptiTrack & f'scan_id = "{scansi}"' & f'recording_id LIKE "%{eye}%"').fetch('KEY')
eye = 'right'
scan_key_right = (pupil_tracking.PupilRotationOptiTrack & f'scan_id = "{scansi}"' & f'recording_id LIKE "%{eye}%"').fetch('KEY')

In [ ]:
scan_key_left

In [ ]:
pupil_left = (pupil_tracking.PupilRotationOptiTrack & scan_key_left).fetch('diameter_opt')[0]
pupil_right = (pupil_tracking.PupilRotationOptiTrack & scan_key_right).fetch('diameter_opt')[0]

# Stagger plots and scale from 0 to 99.9 percentiles
plt.figure(figsize=(12, 4))
plt.plot(pupil_left, label='Left Eye', alpha=0.7)
plt.plot(pupil_right + np.nanmax(pupil_left) * 0.1, label='Right Eye', alpha=0.7)
plt.legend()
plt.xlabel('Frame')
plt.ylabel('Diameter (OptiTrack)')
plt.title(f'Pupil Diameter - {scansi}')
plt.ylim([np.nanpercentile(pupil_left, 0), np.nanpercentile(pupil_left, 99.999)])
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
pupil_tracking.GazeReconstruction3D()

## Visualize Gaze in Head and Gaze in Space

In [ ]:
# Fetch gaze data for a specific scan
scansi = "scan9FU1BEUM"  # Change this to your scan ID
eye = 'left'  # or 'right'

# Get the key for the gaze reconstruction
gaze_key = (pupil_tracking.GazeReconstruction3D & f'scan_id = "{scansi}"' & f'recording_id LIKE "%{eye}%"').fetch('KEY')

if len(gaze_key) > 0:
    gaze_key = gaze_key[0]
    gaze_in_head = (pupil_tracking.GazeReconstruction3D & gaze_key).fetch1('gaze_in_head')
    gaze_in_space = (pupil_tracking.GazeReconstruction3D & gaze_key).fetch1('gaze_in_space')
    torsion = (pupil_tracking.GazeReconstruction3D & gaze_key).fetch1('torsion')
    
    print(f"Gaze data loaded for {scansi}, {eye} eye")
    print(f"Gaze in head shape: {gaze_in_head.shape}")
    print(f"Gaze in space shape: {gaze_in_space.shape}")
else:
    print(f"No gaze reconstruction data found for {scansi}, {eye} eye")

In [ ]:
# Visualize Gaze Direction Components Over Time
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)

# Gaze in Head (left column)
axes[0, 0].plot(gaze_in_head[:, 0], alpha=0.7, linewidth=0.5)
axes[0, 0].set_ylabel('X component')
axes[0, 0].set_title('Gaze in Head')
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(gaze_in_head[:, 1], alpha=0.7, linewidth=0.5, color='orange')
axes[1, 0].set_ylabel('Y component')
axes[1, 0].grid(True, alpha=0.3)

axes[2, 0].plot(gaze_in_head[:, 2], alpha=0.7, linewidth=0.5, color='green')
axes[2, 0].set_ylabel('Z component')
axes[2, 0].set_xlabel('Frame')
axes[2, 0].grid(True, alpha=0.3)

# Gaze in Space (right column)
axes[0, 1].plot(gaze_in_space[:, 0], alpha=0.7, linewidth=0.5)
axes[0, 1].set_ylabel('X component')
axes[0, 1].set_title('Gaze in Space')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(gaze_in_space[:, 1], alpha=0.7, linewidth=0.5, color='orange')
axes[1, 1].set_ylabel('Y component')
axes[1, 1].grid(True, alpha=0.3)

axes[2, 1].plot(gaze_in_space[:, 2], alpha=0.7, linewidth=0.5, color='green')
axes[2, 1].set_ylabel('Z component')
axes[2, 1].set_xlabel('Frame')
axes[2, 1].grid(True, alpha=0.3)

plt.suptitle(f'Gaze Direction Components - {scansi} ({eye} eye)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Convert gaze vectors to azimuth/elevation angles for visualization
def gaze_to_angles(gaze_vectors):
    """Convert 3D gaze unit vectors to azimuth and elevation angles in degrees."""
    x, y, z = gaze_vectors[:, 0], gaze_vectors[:, 1], gaze_vectors[:, 2]
    
    # Azimuth (horizontal angle) - angle in XY plane from Y axis
    azimuth = np.degrees(np.arctan2(x, y))
    
    # Elevation (vertical angle) - angle from XY plane
    elevation = np.degrees(np.arcsin(np.clip(z, -1, 1)))
    
    return azimuth, elevation

# Calculate angles for both gaze types
az_head, el_head = gaze_to_angles(gaze_in_head)
az_space, el_space = gaze_to_angles(gaze_in_space)

# Plot azimuth and elevation comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Azimuth comparison
axes[0].plot(az_head, alpha=0.7, linewidth=0.5, label='Gaze in Head')
axes[0].plot(az_space, alpha=0.7, linewidth=0.5, label='Gaze in Space')
axes[0].set_ylabel('Azimuth (degrees)')
axes[0].set_title(f'Gaze Azimuth Comparison - {scansi} ({eye} eye)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([np.nanpercentile(np.concatenate([az_head, az_space]), 0.1), 
                  np.nanpercentile(np.concatenate([az_head, az_space]), 99.9)])

# Elevation comparison
axes[1].plot(el_head, alpha=0.7, linewidth=0.5, label='Gaze in Head')
axes[1].plot(el_space, alpha=0.7, linewidth=0.5, label='Gaze in Space')
axes[1].set_ylabel('Elevation (degrees)')
axes[1].set_xlabel('Frame')
axes[1].set_title('Gaze Elevation Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([np.nanpercentile(np.concatenate([el_head, el_space]), 0.1), 
                  np.nanpercentile(np.concatenate([el_head, el_space]), 99.9)])

plt.tight_layout()
plt.show()

In [ ]:
# 2D scatter plot: Gaze in Head vs Gaze in Space (Azimuth-Elevation space)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subsample for cleaner visualization (every 10th frame)
subsample = 10
valid_idx = np.where(np.isfinite(az_head) & np.isfinite(el_head) & np.isfinite(az_space) & np.isfinite(el_space))[0]
valid_idx_sub = valid_idx[::subsample]

# Color by time
colors = np.arange(len(valid_idx_sub))

# Gaze in Head
sc1 = axes[0].scatter(az_head[valid_idx_sub], el_head[valid_idx_sub], 
                       c=colors, cmap='viridis', s=2, alpha=0.5)
axes[0].set_xlabel('Azimuth (degrees)')
axes[0].set_ylabel('Elevation (degrees)')
axes[0].set_title('Gaze in Head')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='--', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='--', linewidth=0.5)

# Gaze in Space
sc2 = axes[1].scatter(az_space[valid_idx_sub], el_space[valid_idx_sub], 
                       c=colors, cmap='viridis', s=2, alpha=0.5)
axes[1].set_xlabel('Azimuth (degrees)')
axes[1].set_ylabel('Elevation (degrees)')
axes[1].set_title('Gaze in Space')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='--', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='--', linewidth=0.5)

plt.colorbar(sc2, ax=axes[1], label='Time (frame index, subsampled)')
plt.suptitle(f'Gaze Distribution - {scansi} ({eye} eye)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3D Visualization of Gaze Vectors on a Unit Sphere
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 6))

# Subsample for 3D plot
subsample_3d = 50
valid_idx = np.where(np.all(np.isfinite(gaze_in_head), axis=1) & np.all(np.isfinite(gaze_in_space), axis=1))[0]
valid_idx_sub = valid_idx[::subsample_3d]

# Create unit sphere wireframe
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, np.pi, 20)
x_sphere = 0.95 * np.outer(np.cos(u), np.sin(v))
y_sphere = 0.95 * np.outer(np.sin(u), np.sin(v))
z_sphere = 0.95 * np.outer(np.ones(np.size(u)), np.cos(v))

# Gaze in Head - 3D
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_wireframe(x_sphere, y_sphere, z_sphere, color='lightgray', alpha=0.3, linewidth=0.5)
ax1.scatter(gaze_in_head[valid_idx_sub, 0], 
            gaze_in_head[valid_idx_sub, 1], 
            gaze_in_head[valid_idx_sub, 2], 
            c=np.arange(len(valid_idx_sub)), cmap='plasma', s=5, alpha=0.6)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Gaze in Head (Unit Sphere)')
ax1.set_box_aspect([1, 1, 1])

# Gaze in Space - 3D
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_wireframe(x_sphere, y_sphere, z_sphere, color='lightgray', alpha=0.3, linewidth=0.5)
sc = ax2.scatter(gaze_in_space[valid_idx_sub, 0], 
                 gaze_in_space[valid_idx_sub, 1], 
                 gaze_in_space[valid_idx_sub, 2], 
                 c=np.arange(len(valid_idx_sub)), cmap='plasma', s=5, alpha=0.6)
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('Gaze in Space (Unit Sphere)')
ax2.set_box_aspect([1, 1, 1])

plt.suptitle(f'3D Gaze Vectors - {scansi} ({eye} eye)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare both eyes: Gaze in Head vs Gaze in Space
scansi = "scan9FU1BEUM"  # Change this to your scan ID

# Fetch both eyes
gaze_key_left = (pupil_tracking.GazeReconstruction3D & f'scan_id = "{scansi}"' & 'recording_id LIKE "%left%"').fetch('KEY')
gaze_key_right = (pupil_tracking.GazeReconstruction3D & f'scan_id = "{scansi}"' & 'recording_id LIKE "%right%"').fetch('KEY')

if len(gaze_key_left) > 0 and len(gaze_key_right) > 0:
    # Left eye
    gaze_in_head_left = (pupil_tracking.GazeReconstruction3D & gaze_key_left[0]).fetch1('gaze_in_head')
    gaze_in_space_left = (pupil_tracking.GazeReconstruction3D & gaze_key_left[0]).fetch1('gaze_in_space')
    
    # Right eye
    gaze_in_head_right = (pupil_tracking.GazeReconstruction3D & gaze_key_right[0]).fetch1('gaze_in_head')
    gaze_in_space_right = (pupil_tracking.GazeReconstruction3D & gaze_key_right[0]).fetch1('gaze_in_space')
    
    # Calculate angles
    az_head_left, el_head_left = gaze_to_angles(gaze_in_head_left)
    az_space_left, el_space_left = gaze_to_angles(gaze_in_space_left)
    az_head_right, el_head_right = gaze_to_angles(gaze_in_head_right)
    az_space_right, el_space_right = gaze_to_angles(gaze_in_space_right)
    
    # Plot comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Azimuth in Head
    axes[0, 0].plot(az_head_left, alpha=0.7, linewidth=0.5, label='Left Eye')
    axes[0, 0].plot(az_head_right, alpha=0.7, linewidth=0.5, label='Right Eye')
    axes[0, 0].set_ylabel('Azimuth (degrees)')
    axes[0, 0].set_title('Gaze in Head - Azimuth')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Elevation in Head
    axes[1, 0].plot(el_head_left, alpha=0.7, linewidth=0.5, label='Left Eye')
    axes[1, 0].plot(el_head_right, alpha=0.7, linewidth=0.5, label='Right Eye')
    axes[1, 0].set_ylabel('Elevation (degrees)')
    axes[1, 0].set_xlabel('Frame')
    axes[1, 0].set_title('Gaze in Head - Elevation')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Azimuth in Space
    axes[0, 1].plot(az_space_left, alpha=0.7, linewidth=0.5, label='Left Eye')
    axes[0, 1].plot(az_space_right, alpha=0.7, linewidth=0.5, label='Right Eye')
    axes[0, 1].set_ylabel('Azimuth (degrees)')
    axes[0, 1].set_title('Gaze in Space - Azimuth')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Elevation in Space
    axes[1, 1].plot(el_space_left, alpha=0.7, linewidth=0.5, label='Left Eye')
    axes[1, 1].plot(el_space_right, alpha=0.7, linewidth=0.5, label='Right Eye')
    axes[1, 1].set_ylabel('Elevation (degrees)')
    axes[1, 1].set_xlabel('Frame')
    axes[1, 1].set_title('Gaze in Space - Elevation')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'Both Eyes Gaze Comparison - {scansi}', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f"Could not find data for both eyes for scan {scansi}")

In [ ]:
# Heatmap visualization of gaze distribution
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Filter valid data
valid_head = np.isfinite(az_head) & np.isfinite(el_head)
valid_space = np.isfinite(az_space) & np.isfinite(el_space)

# Gaze in Head heatmap
if np.sum(valid_head) > 100:
    xy_head = np.vstack([az_head[valid_head], el_head[valid_head]])
    kde_head = gaussian_kde(xy_head)
    
    # Create grid
    xmin, xmax = np.percentile(az_head[valid_head], [1, 99])
    ymin, ymax = np.percentile(el_head[valid_head], [1, 99])
    xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
    positions = np.vstack([xx.ravel(), yy.ravel()])
    z_head = np.reshape(kde_head(positions).T, xx.shape)
    
    axes[0].imshow(np.rot90(z_head), cmap='hot', extent=[xmin, xmax, ymin, ymax], aspect='auto')
    axes[0].set_xlabel('Azimuth (degrees)')
    axes[0].set_ylabel('Elevation (degrees)')
    axes[0].set_title('Gaze in Head - Density Heatmap')

# Gaze in Space heatmap  
if np.sum(valid_space) > 100:
    xy_space = np.vstack([az_space[valid_space], el_space[valid_space]])
    kde_space = gaussian_kde(xy_space)
    
    xmin, xmax = np.percentile(az_space[valid_space], [1, 99])
    ymin, ymax = np.percentile(el_space[valid_space], [1, 99])
    xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
    positions = np.vstack([xx.ravel(), yy.ravel()])
    z_space = np.reshape(kde_space(positions).T, xx.shape)
    
    axes[1].imshow(np.rot90(z_space), cmap='hot', extent=[xmin, xmax, ymin, ymax], aspect='auto')
    axes[1].set_xlabel('Azimuth (degrees)')
    axes[1].set_ylabel('Elevation (degrees)')
    axes[1].set_title('Gaze in Space - Density Heatmap')

plt.suptitle(f'Gaze Distribution Heatmaps - {scansi} ({eye} eye)', fontsize=14)
plt.tight_layout()
plt.show()

## 3D Animation: Mouse Head with Binocular Gaze Vectors
Combines rigid body tracking (head position, orientation) with left and right eye gaze vectors.

In [ ]:
# Load rigid body tracking data and both eyes gaze data
scansi = "scan9FU1BEUM"  # Change to your scan ID

# Get keys for both eyes
gaze_key_left = (pupil_tracking.GazeReconstruction3D & f'scan_id = "{scansi}"' & 'recording_id LIKE "%left%"').fetch('KEY')
gaze_key_right = (pupil_tracking.GazeReconstruction3D & f'scan_id = "{scansi}"' & 'recording_id LIKE "%right%"').fetch('KEY')

if len(gaze_key_left) > 0 and len(gaze_key_right) > 0:
    # Fetch gaze data
    gaze_in_head_left = (pupil_tracking.GazeReconstruction3D & gaze_key_left[0]).fetch1('gaze_in_head')
    gaze_in_space_left = (pupil_tracking.GazeReconstruction3D & gaze_key_left[0]).fetch1('gaze_in_space')
    gaze_in_head_right = (pupil_tracking.GazeReconstruction3D & gaze_key_right[0]).fetch1('gaze_in_head')
    gaze_in_space_right = (pupil_tracking.GazeReconstruction3D & gaze_key_right[0]).fetch1('gaze_in_space')
    
    # Fetch rigid body tracking data (use key from one of the gaze entries)
    rigid_key = {k: v for k, v in gaze_key_left[0].items() if k in ['session_id', 'scan_id', 'subject']}
    
    # Get eye positions and head orientation
    eye_left_global = (virtual_markers_optitrack.RigidMouseTracking & rigid_key).fetch1('eye_left_global')
    eye_right_global = (virtual_markers_optitrack.RigidMouseTracking & rigid_key).fetch1('eye_right_global')
    nose_global = (virtual_markers_optitrack.RigidMouseTracking & rigid_key).fetch1('nose_global')
    pivot_center = (virtual_markers_optitrack.RigidMouseTracking & rigid_key).fetch1('pivot_eye_center_global')
    
    print(f"Loaded data for scan: {scansi}")
    print(f"Gaze left shape: {gaze_in_space_left.shape}")
    print(f"Gaze right shape: {gaze_in_space_right.shape}")
    print(f"Eye left global shape: {eye_left_global.shape}")
    print(f"Eye right global shape: {eye_right_global.shape}")
else:
    print(f"Could not find gaze data for both eyes for scan {scansi}")

In [ ]:
# Create 3D animation of mouse head with binocular gaze vectors
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

# Animation parameters
start_frame = 0
num_frames_anim = 500  # Number of frames to animate
frame_skip = 5  # Skip frames for smoother animation (reduces file size)
gaze_length = 0.02  # Length of gaze vectors in world units

# Ensure we don't exceed available frames
n_frames = min(gaze_in_space_left.shape[0], eye_left_global.shape[0])
end_frame = min(start_frame + num_frames_anim * frame_skip, n_frames)
frame_indices = np.arange(start_frame, end_frame, frame_skip)

print(f"Animating {len(frame_indices)} frames (from {start_frame} to {end_frame}, skip {frame_skip})")

# Find valid frames where all data is available
valid_frames = []
for i in frame_indices:
    if (np.all(np.isfinite(eye_left_global[i])) and 
        np.all(np.isfinite(eye_right_global[i])) and
        np.all(np.isfinite(gaze_in_space_left[i])) and
        np.all(np.isfinite(gaze_in_space_right[i])) and
        np.all(np.isfinite(nose_global[i]))):
        valid_frames.append(i)

print(f"Valid frames for animation: {len(valid_frames)}")

# Compute axis limits from valid data
all_positions = np.vstack([
    eye_left_global[valid_frames],
    eye_right_global[valid_frames],
    nose_global[valid_frames]
])
center = np.mean(all_positions, axis=0)
max_range = np.max(np.ptp(all_positions, axis=0)) / 2 * 1.5

xlim = [center[0] - max_range, center[0] + max_range]
ylim = [center[1] - max_range, center[1] + max_range]
zlim = [center[2] - max_range, center[2] + max_range]

In [ ]:
# Create the animation
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

def init():
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    return []

def update(frame_num):
    ax.cla()
    
    i = valid_frames[frame_num]
    
    # Get positions
    eye_l = eye_left_global[i]
    eye_r = eye_right_global[i]
    nose = nose_global[i]
    head_center = pivot_center[i]
    
    # Get gaze directions
    gaze_l = gaze_in_space_left[i]
    gaze_r = gaze_in_space_right[i]
    
    # Draw head triangle (eyes + nose)
    head_triangle = np.array([eye_l, eye_r, nose, eye_l])
    ax.plot(head_triangle[:, 0], head_triangle[:, 1], head_triangle[:, 2], 
            'k-', linewidth=2, label='Head')
    
    # Draw eye positions
    ax.scatter(*eye_l, c='blue', s=100, marker='o', label='Left Eye')
    ax.scatter(*eye_r, c='red', s=100, marker='o', label='Right Eye')
    ax.scatter(*nose, c='green', s=80, marker='^', label='Nose')
    ax.scatter(*head_center, c='purple', s=60, marker='x', label='Head Center')
    
    # Draw gaze vectors from eye positions
    gaze_end_l = eye_l + gaze_l * gaze_length
    gaze_end_r = eye_r + gaze_r * gaze_length
    
    ax.quiver(eye_l[0], eye_l[1], eye_l[2], 
              gaze_l[0]*gaze_length, gaze_l[1]*gaze_length, gaze_l[2]*gaze_length,
              color='blue', arrow_length_ratio=0.15, linewidth=2, alpha=0.8)
    ax.quiver(eye_r[0], eye_r[1], eye_r[2], 
              gaze_r[0]*gaze_length, gaze_r[1]*gaze_length, gaze_r[2]*gaze_length,
              color='red', arrow_length_ratio=0.15, linewidth=2, alpha=0.8)
    
    # Draw trajectory trail (last 20 frames)
    trail_length = min(20, frame_num)
    if trail_length > 1:
        trail_frames = valid_frames[max(0, frame_num-trail_length):frame_num+1]
        trail_centers = pivot_center[trail_frames]
        ax.plot(trail_centers[:, 0], trail_centers[:, 1], trail_centers[:, 2], 
                'gray', alpha=0.5, linewidth=1)
    
    # Set axis properties
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_zlabel('Z (m)')
    ax.set_title(f'Mouse Head & Gaze - {scansi}\nFrame: {i}')
    ax.legend(loc='upper left', fontsize=8)
    
    # Set view angle
    ax.view_init(elev=20, azim=45 + frame_num * 0.5)  # Slowly rotate view
    
    return []

# Create animation
anim = animation.FuncAnimation(fig, update, init_func=init, 
                                frames=len(valid_frames), 
                                interval=50, blit=False)

plt.close()  # Prevent static figure display
print("Animation created. Displaying...")

In [ ]:
# Display animation in notebook (requires ffmpeg or pillow)
HTML(anim.to_jshtml())

In [ ]:
# Save animation as MP4 file
save_dir = os.path.expanduser('~/figures')
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, f'{scansi}_binocular_gaze_animation.mp4')

print(f"Saving animation to {save_path}...")
anim.save(save_path, writer='ffmpeg', fps=20, dpi=100)
print(f"Animation saved to: {save_path}")

### Dual View Animation: Gaze in Head vs Gaze in Space
Side-by-side comparison showing gaze vectors in head-centered and world-centered coordinates.

In [ ]:
# Dual view animation: Gaze in Head (left) vs Gaze in Space (right)
fig_dual = plt.figure(figsize=(16, 8))

# Create unit sphere for gaze visualization
u_sphere = np.linspace(0, 2 * np.pi, 30)
v_sphere = np.linspace(0, np.pi, 20)
x_unit = 0.9 * np.outer(np.cos(u_sphere), np.sin(v_sphere))
y_unit = 0.9 * np.outer(np.sin(u_sphere), np.sin(v_sphere))
z_unit = 0.9 * np.outer(np.ones(np.size(u_sphere)), np.cos(v_sphere))

def init_dual():
    return []

def update_dual(frame_num):
    fig_dual.clear()
    
    i = valid_frames[frame_num]
    
    # --- Left subplot: Gaze in Head (unit sphere) ---
    ax1 = fig_dual.add_subplot(121, projection='3d')
    
    # Draw unit sphere
    ax1.plot_wireframe(x_unit, y_unit, z_unit, color='lightgray', alpha=0.2, linewidth=0.3)
    
    # Get gaze in head for both eyes
    gaze_head_l = gaze_in_head_left[i]
    gaze_head_r = gaze_in_head_right[i]
    
    # Draw gaze vectors from origin
    ax1.quiver(0, 0, 0, gaze_head_l[0], gaze_head_l[1], gaze_head_l[2],
               color='blue', arrow_length_ratio=0.1, linewidth=3, label='Left Eye')
    ax1.quiver(0, 0, 0, gaze_head_r[0], gaze_head_r[1], gaze_head_r[2],
               color='red', arrow_length_ratio=0.1, linewidth=3, label='Right Eye')
    
    # Draw gaze points on sphere
    ax1.scatter(*gaze_head_l, c='blue', s=100, marker='o')
    ax1.scatter(*gaze_head_r, c='red', s=100, marker='o')
    
    # Trail of recent gaze positions (last 50 frames)
    trail_len = min(50, frame_num)
    if trail_len > 1:
        trail_idx = valid_frames[max(0, frame_num-trail_len):frame_num+1]
        trail_l = gaze_in_head_left[trail_idx]
        trail_r = gaze_in_head_right[trail_idx]
        valid_trail_l = ~np.any(np.isnan(trail_l), axis=1)
        valid_trail_r = ~np.any(np.isnan(trail_r), axis=1)
        if np.any(valid_trail_l):
            ax1.plot(trail_l[valid_trail_l, 0], trail_l[valid_trail_l, 1], trail_l[valid_trail_l, 2], 
                     'b-', alpha=0.3, linewidth=1)
        if np.any(valid_trail_r):
            ax1.plot(trail_r[valid_trail_r, 0], trail_r[valid_trail_r, 1], trail_r[valid_trail_r, 2], 
                     'r-', alpha=0.3, linewidth=1)
    
    ax1.set_xlim([-1.2, 1.2])
    ax1.set_ylim([-1.2, 1.2])
    ax1.set_zlim([-1.2, 1.2])
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y (Forward)')
    ax1.set_zlabel('Z (Up)')
    ax1.set_title(f'Gaze in Head\nFrame: {i}')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.view_init(elev=20, azim=45)
    
    # --- Right subplot: Gaze in Space (with head position) ---
    ax2 = fig_dual.add_subplot(122, projection='3d')
    
    # Get positions
    eye_l = eye_left_global[i]
    eye_r = eye_right_global[i]
    nose = nose_global[i]
    
    # Get gaze directions
    gaze_space_l = gaze_in_space_left[i]
    gaze_space_r = gaze_in_space_right[i]
    
    # Draw head triangle
    head_tri = np.array([eye_l, eye_r, nose, eye_l])
    ax2.plot(head_tri[:, 0], head_tri[:, 1], head_tri[:, 2], 'k-', linewidth=2)
    
    # Draw eye positions
    ax2.scatter(*eye_l, c='blue', s=100, marker='o')
    ax2.scatter(*eye_r, c='red', s=100, marker='o')
    ax2.scatter(*nose, c='green', s=80, marker='^')
    
    # Draw gaze vectors in space
    ax2.quiver(eye_l[0], eye_l[1], eye_l[2],
               gaze_space_l[0]*gaze_length, gaze_space_l[1]*gaze_length, gaze_space_l[2]*gaze_length,
               color='blue', arrow_length_ratio=0.15, linewidth=2, alpha=0.9)
    ax2.quiver(eye_r[0], eye_r[1], eye_r[2],
               gaze_space_r[0]*gaze_length, gaze_space_r[1]*gaze_length, gaze_space_r[2]*gaze_length,
               color='red', arrow_length_ratio=0.15, linewidth=2, alpha=0.9)
    
    # Trail of head position
    if trail_len > 1:
        trail_centers = pivot_center[valid_frames[max(0, frame_num-trail_len):frame_num+1]]
        valid_trail = ~np.any(np.isnan(trail_centers), axis=1)
        if np.any(valid_trail):
            ax2.plot(trail_centers[valid_trail, 0], trail_centers[valid_trail, 1], 
                     trail_centers[valid_trail, 2], 'gray', alpha=0.4, linewidth=1)
    
    ax2.set_xlim(xlim)
    ax2.set_ylim(ylim)
    ax2.set_zlim(zlim)
    ax2.set_xlabel('X (m)')
    ax2.set_ylabel('Y (m)')
    ax2.set_zlabel('Z (m)')
    ax2.set_title(f'Gaze in Space\nFrame: {i}')
    ax2.view_init(elev=20, azim=45 + frame_num * 0.3)
    
    fig_dual.suptitle(f'{scansi} - Binocular Gaze Comparison', fontsize=14)
    plt.tight_layout()
    
    return []

# Create dual animation
anim_dual = animation.FuncAnimation(fig_dual, update_dual, init_func=init_dual,
                                     frames=len(valid_frames),
                                     interval=50, blit=False)

plt.close()
print("Dual view animation created.")

In [ ]:
# Display dual animation in notebook
HTML(anim_dual.to_jshtml())

In [ ]:
# Save dual animation as MP4
save_path_dual = os.path.join(save_dir, f'{scansi}_dual_gaze_animation.mp4')
print(f"Saving dual animation to {save_path_dual}...")
anim_dual.save(save_path_dual, writer='ffmpeg', fps=20, dpi=100)
print(f"Dual animation saved to: {save_path_dual}")

In [ ]:
virtual_markers_optitrack.RigidMouseTracking()

In [ ]:
# Gaze Projection Animation - Binocular gaze rays from head into space
import sys
sys.path.insert(0, '.')
from gaze_projection_visualization import create_gaze_projection_animation, display_animation, save_animation

# Create the gaze projection animation
anim_proj, fig_proj = create_gaze_projection_animation(
    eye_left_global, eye_right_global, nose_global, pivot_center,
    gaze_in_space_left, gaze_in_space_right,
    valid_frames, scansi, ray_length=0.1
)

In [ ]:
# Display the gaze projection animation
display_animation(anim_proj)

In [ ]:
# Visualize schema relationships
import warnings
warnings.filterwarnings('ignore')
diagram = dj.Diagram(pupil_tracking.schema) + dj.Diagram(virtual_markers_optitrack.schema) -1
diagram

In [ ]:
pupil_tracking.schema.list_tables()

In [ ]:

# assuming you already did something like:
# import pupil_tracking
# pupil_tracking.schema

conn = dj.conn()
db = pupil_tracking.schema.database        # e.g. 'roselab_pupil_tracking'
table_name = '#denoising_method'   # the one you want to drop

t = dj.FreeTable(conn, f'`{db}`.`{table_name}`')

# Safe drop with dependency checks + confirmation prompt
# t.drop()

# (or, if you really want to force it without cascade / prompt)
# t.drop_quick()

In [ ]:
virtual_markers_optitrack.schema.list_tables()

In [ ]:

# assuming you already did something like:
# import pupil_tracking
# pupil_tracking.schemas

conn = dj.conn()
db = virtual_markers_optitrack.schema.database        # e.g. 'roselab_pupil_tracking'
table_name = 'eye_nose_cam_pos_calib_test'   # the one you want to drop

t = dj.FreeTable(conn, f'`{db}`.`{table_name}`')

# Safe drop with dependency checks + confirmation prompt
# t.drop()

# (or, if you really want to force it without cascade / prompt)
# t.drop_quick()

In [ ]:
t.drop()

In [ ]:
t.drop()

In [ ]:
pupil_tracking.PupilEllipseFitting.populate(process_keys, display_progress = True, suppress_errors=True) #REMOVE [100] FOR ALL

In [ ]:
process_keys[100]['session_id']

In [ ]:
whicheye = 'left'
# whicheye = 'right'
# scan_id = process_keys[100]['scan_id'] 
scan_id = "scan9FU07CC2"
# scan_key = (pupil_tracking.PupilEllipseFitting & f'scan_id = "{scan_id}"').fetch('KEY')
scan_key = (scan.Scan & 'scan_id = "scan9FU07CC2"').fetch('KEY')[0]
dlc_scan_key = (model.PoseEstimationNew & scan_key & f'recording_id LIKE "%{whicheye}%"').fetch1('KEY')
# dlc_scan_key = (model.PoseEstimationNew & scan_key & f'recording_id LIKE "%face%"').fetch1('KEY')
# dlc_scan_key = (model.PoseEstimationNew & scan_key ).fetch('KEY')

In [ ]:
pupil_tracking.PupilEllipseFitting & scan_key

In [ ]:
 diameter = (pupil_tracking.PupilEllipseFitting & scan_key).fetch1('diameter')
diameter

In [ ]:
colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']

plt.figure()
for i, key in enumerate(scan_key):
    diameter = (pupil_tracking.PupilEllipseFitting & key).fetch1('diameter')
    plt.plot(diameter, color=colors[i % len(colors)], label=f'recording_id: {key["recording_id"]}')

plt.xlabel('Frame')
plt.ylabel('Diameter')
plt.title('Pupil Diameter Over Time')
plt.ylim([np.nanpercentile(diameter, 0) , np.nanpercentile(diameter, 99.9) ])
#plt.xlim([0, 500])
plt.legend()
plt.show()


In [ ]:
pupil_tracking.PupilEllipseFitting & scan_key

In [ ]:
angle_type = 'phi'  # Change to 'theta' if needed

plt.figure()
for i, key in enumerate(scan_key):
    angle = (pupil_tracking.PupilEllipseFitting & key).fetch1(angle_type)
    plt.plot(angle, color=colors[i % len(colors)], label=f'recording_id: {key["recording_id"]}')

plt.xlabel('Frame')
plt.ylabel('Angle')
plt.title(f'Pupil {angle_type.capitalize()} Over Time')
plt.ylim([np.pi/2, -np.pi/2])
plt.legend()
plt.show()


load video

In [ ]:
from tqdm import tqdm
import cv2

def read_video_cv2(vid, full_video=False, n_frames=1000):
    cap = cv2.VideoCapture(vid)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    all = []
    i = 0
    if full_video == True:
        n_frames = total_frames
    for i in tqdm(range(n_frames), desc="Loading video"):
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        arr = np.array(frame_rgb)
        all.append(arr)
    return np.array(all)

videofile = (model.VideoRecordingNew * model.VideoRecordingNew.File & dlc_scan_key ).fetch1('file_path')

num_frames=1000

v_frames = read_video_cv2(videofile, n_frames = num_frames)

In [ ]:
# v_frames = np.flipud(v_frames)

In [ ]:
ellipsedict = (pupil_tracking.PupilEllipseFitting & dlc_scan_key).fetch('ellipse_dict')[0]

In [ ]:
df=model.PoseEstimationNew.get_trajectory(dlc_scan_key)
df_xy = df.iloc[:,df.columns.get_level_values(2).isin(["x","y"])][dlc_scan_key['model_name']]
df_flat = df_xy.copy()
df_flat.columns = df_flat.columns.map('_'.join)

ir_x = df_flat['IR_x'].values
ir_y = df_flat['IR_y'].values

In [ ]:
cam_cent = (pupil_tracking.PupilEllipseFitting & dlc_scan_key).fetch('cam_center')[0]

In [ ]:
(pupil_tracking.PupilEllipseFitting & dlc_scan_key)

#### make movie of all frames

displax with slider

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from tqdm import tqdm

# Generate overlay frames
overlay_frames = []

for i in tqdm(range(num_frames), desc="Generating overlay frames"):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(v_frames[i])
    
    if i < len(ellipsedict) and not np.isnan(ellipsedict[i]['a']):
        a = ellipsedict[i]['a']
        b = ellipsedict[i]['b']
        X0_in = ellipsedict[i]['X0_in']
        Y0_in = ellipsedict[i]['Y0_in']
        angle = ellipsedict[i]['phi']
        
        t = np.linspace(0, 2 * np.pi, 100)
        
        x2 = a * np.cos(t) * np.cos(angle) - b * np.sin(t) * np.sin(angle) + X0_in
        y2 = a * np.cos(t) * np.sin(angle) + b * np.sin(t) * np.cos(angle) + Y0_in
        
        ax.plot(x2 + ir_x[i], y2 + ir_y[i], linewidth=2)
        ax.plot(cam_cent[0] + ir_x[i], cam_cent[1] + ir_y[i], '.', color='red', markersize=6)
    
    ax.axis("off")
    fig.canvas.draw()
    
    # Convert figure to numpy array
    overlay_frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    overlay_frame = overlay_frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    overlay_frames.append(overlay_frame)
    plt.close(fig)

# Function to update the displayed frame
def update_frame(frame_index):
    plt.figure(figsize=(5, 5))
    plt.imshow(overlay_frames[frame_index])
    plt.axis("off")
    plt.show()

# Create a slider widget
frame_slider = widgets.IntSlider(min=0, max=num_frames-1, step=1, value=0)
widgets.interact(update_frame, frame_index=frame_slider)

# Display the slider
display(frame_slider)


save movie

In [ ]:
import matplotlib.animation as animation
from tqdm import tqdm
import os

# Ensure the directory exists
save_dir = os.path.expanduser('~/figures')
os.makedirs(save_dir, exist_ok=True)

num_frames = 1000

fig = plt.figure(figsize=(5, 5))

def update(i):
    plt.clf()
    id = i  # Frame index directly corresponds to `i`
    
    plt.imshow(v_frames[id])  # Display the current frame
    
    # Check for valid ellipse data
    if id < len(ellipsedict) and not np.isnan(ellipsedict[id]['a']):
        # Extract ellipse parameters
        a = ellipsedict[id]['a']
        b = ellipsedict[id]['b']
        X0_in = ellipsedict[id]['X0_in']
        Y0_in = ellipsedict[id]['Y0_in']
        angle = ellipsedict[id]['phi']
        
        t = np.linspace(0, 2 * np.pi, 100)
        
        # Parametric equation for the ellipse
        x2 = a * np.cos(t) * np.cos(angle) - b * np.sin(t) * np.sin(angle) + X0_in
        y2 = a * np.cos(t) * np.sin(angle) + b * np.sin(t) * np.cos(angle) + Y0_in
        
        # Plot ellipse and relevant points
        plt.plot(x2 + ir_x[id], y2 + ir_y[id], linewidth=2)
        plt.plot(cam_cent[0] + ir_x[id], cam_cent[1] + ir_y[id], '.', color='red', markersize=6)
    
    plt.axis("off")
    plt.tight_layout()

# Progress bar integration
pbar = tqdm(total=num_frames)

def update_with_progress(i):
    update(i)
    pbar.update()

# Create animation
ani = animation.FuncAnimation(fig, update_with_progress, frames=range(0, num_frames - 1), repeat=False)

# Use recording_id from scan_key as filename
recording_id = dlc_scan_key['recording_id']
save_path = os.path.join(save_dir, f'{recording_id}_movie.mp4')
ani.save(save_path, writer='ffmpeg', fps=60)
pbar.close()

print(save_path)

In [ ]:

print(save_dir)

In [ ]:


session.Session & scan_key


In [ ]:
plt.figure(figsize=(20, 20))

start_fr = 0
for i in range(0, num_frames - 1):
    id = i + start_fr
    plt.subplot(10, 10, i + 1)
    
    plt.imshow(v_frames[id])
    
    # Extract ellipse parameters from ellipsedict for frame 'id'
    if not np.isnan(ellipsedict[id]['a']):  # Avoid NaN entries
        a = ellipsedict[id]['a']
        b = ellipsedict[id]['b']
        X0_in = ellipsedict[id]['X0_in']
        Y0_in = ellipsedict[id]['Y0_in']
        angle = ellipsedict[id]['phi']  # Angle phi to rotate the ellipse
        
        t = np.linspace(0, 2 * np.pi, 100)
        
        # Parametric equation for the ellipse
        x2 = a * np.cos(t) * np.cos(angle) - b * np.sin(t) * np.sin(angle) + X0_in
        y2 = a * np.cos(t) * np.sin(angle) + b * np.sin(t) * np.cos(angle) + Y0_in
        
        plt.plot(x2 + ir_x[id], y2 + ir_y[id])  # Adjust ellipse by offsets
        # plt.plot(pc_x[id] + ir_x[id], pc_y[id] + ir_y[id], '.')
        plt.plot(cam_cent[0] + ir_x[id], cam_cent[1] + ir_y[id], '.', color='red')
    
    plt.axis("off")
    plt.tight_layout()